# judgeman: fine-tune a small judge (Leg 2)

Trains a LoRA adapter on Qwen 3.5 so it judges agent steps better, then has it answer the exam prompts.

**Before running**
1. Right panel → *Session options*: Accelerator **GPU T4 x2** (or P100), Internet **On**.
2. *Add input* → attach the dataset you uploaded (it holds `train.jsonl`, `val.jsonl`, `exam.jsonl`).
3. First run: leave `TRIAL = True` below. It takes about 15 minutes and proves everything works.
4. Real run: set `TRIAL = False`, then *Save Version → Save & Run All (Commit)* so it keeps running with the tab closed.

**Afterwards** download `exam-replies.jsonl` and `exam-replies-base.jsonl` from the Output tab.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
TRIAL = True            # True: tiny run to check the notebook works. False: the real run.
MODEL = "unsloth/Qwen3.5-9B"   # if this runs out of GPU memory, switch to "unsloth/Qwen3.5-4B"
MAX_SEQ = 4096
EPOCHS = 2
RUN_BASE_EXAM = True    # also answer the exam with the untrained model, for a like-for-like baseline

import glob, json, os, time
DATA = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
OUT = "/kaggle/working"
read = lambda name: [json.loads(l) for l in open(f"{DATA}/{name}.jsonl")]
train_rows, val_rows, exam_rows = read("train"), read("val"), read("exam")
if TRIAL:
    train_rows, val_rows, exam_rows = train_rows[:40], val_rows[:10], exam_rows[:30]
print(len(train_rows), "train,", len(val_rows), "val,", len(exam_rows), "exam prompts from", DATA)

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, full_finetuning=False,
)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 2**30:.1f} GB used after loading")

In [ ]:
def render(messages, for_answer):
    # thinking is off: the judge prompt already asks for a short critique before the verdict
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=for_answer, enable_thinking=False
    )

def answer_exam(rows, path):
    """Answer every exam prompt and append to `path`. Safe to re-run: finished prompts are skipped."""
    done = {(r["trajectory_id"], r["step"]) for r in map(json.loads, open(path))} if os.path.exists(path) else set()
    FastLanguageModel.for_inference(model)
    started = time.time()
    with open(path, "a") as f:
        for n, row in enumerate(rows):
            if (row["trajectory_id"], row["step"]) in done:
                continue
            inputs = tokenizer(render(row["messages"], True), return_tensors="pt", truncation=True,
                               max_length=MAX_SEQ).to(model.device)
            t0 = time.time()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=220, do_sample=False)
            reply = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
            f.write(json.dumps({"trajectory_id": row["trajectory_id"], "step": row["step"],
                                "reply": reply, "seconds": round(time.time() - t0, 2)}) + "\n")
            f.flush()
            if n % 25 == 0:
                print(n, "of", len(rows), f"{(time.time() - started) / 60:.0f} min", reply[:90].replace("\n", " "))

if RUN_BASE_EXAM:
    answer_exam(exam_rows, f"{OUT}/exam-replies-base.jsonl")

In [ ]:
from datasets import Dataset

FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407, max_seq_length=MAX_SEQ,
)
to_text = lambda rows: Dataset.from_list([{"text": render(r["messages"], False)} for r in rows])
train_ds, val_ds = to_text(train_rows), to_text(val_rows)
print(train_ds[0]["text"][-600:])

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ,
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS, max_steps=5 if TRIAL else -1,
        learning_rate=1e-4, warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=1 if TRIAL else 10, eval_strategy="no" if TRIAL else "epoch",
        per_device_eval_batch_size=1, save_strategy="no", optim="adamw_8bit",
        seed=3407, output_dir=f"{OUT}/checkpoints", report_to="none",
    ),
)
stats = trainer.train()
print(stats.metrics, f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
model.save_pretrained(f"{OUT}/adapter"); tokenizer.save_pretrained(f"{OUT}/adapter")

In [ ]:
answer_exam(exam_rows, f"{OUT}/exam-replies.jsonl")
print("Done. Download exam-replies.jsonl and exam-replies-base.jsonl from the Output tab.")